Monthly Bloom Mapping

In [1]:
# Imports & Initialization
import ee
import geemap
import numpy as np

ee.Authenticate()
ee.Initialize()
geemap.ee_initialize()

In [2]:
# Study Area
roi_fc = ee.FeatureCollection("GEEassetpath")#"projects/ee-name/assets/region"
roi = roi_fc.geometry()

Map = geemap.Map(center=[10, 80], zoom=5) #[latitude, longitude]

In [5]:
# Time Settings  (MUST be early)
START_YEAR = 2003
END_YEAR = 2020

years = ee.List.sequence(START_YEAR, END_YEAR)
months = ee.List.sequence(1, 12)

timeField = 'system:time_start'


In [6]:
# Helper Functions
def monthly_composite(collection, reducer='mean'):
    """Create monthly composites"""
    def by_year(y):
        def by_month(m):
            img = collection \
                .filter(ee.Filter.calendarRange(y, y, 'year')) \
                .filter(ee.Filter.calendarRange(m, m, 'month'))

            img = img.mean() if reducer == 'mean' else img.max()

            return img.set({
                'year': y,
                'month': m,
                timeField: ee.Date.fromYMD(y, m, 1).millis()
            })
        return months.map(by_month)
    return ee.ImageCollection.fromImages(years.map(by_year).flatten())


def add_time_vars(image):
    years_since = ee.Date(image.get(timeField)) \
        .difference(ee.Date('2003-01-01'), 'year')
    return image.addBands([
        ee.Image(years_since).rename('t').float(),
        ee.Image.constant(1).rename('constant')
    ])


def detrend(collection, dependent):
    independents = ee.List(['constant', 't'])

    trend = collection.select(independents.add(dependent)) \
        .reduce(ee.Reducer.linearRegression(independents.length(), 1))

    coeffs = trend.select('coefficients') \
        .arrayProject([0]) \
        .arrayFlatten([independents])

    def remove_trend(img):
        return img.select(dependent) \
            .subtract(
                img.select(independents)
                .multiply(coeffs)
                .reduce('sum')
            ) \
            .rename(dependent) \
            .copyProperties(img, [timeField])

    return collection.map(remove_trend)

In [7]:
# Chlorophyll (Bloom Detection)
chl = ee.ImageCollection('NASA/OCEANDATA/MODIS-Aqua/L3SMI') \
    .select('chlor_a') \
    .filterDate('2003-01-01', '2020-12-31') \
    .map(lambda i: i.clip(roi))

chl_monthly = monthly_composite(chl, reducer='max')

def bloom_mask(img):
    mask = img.gt(3.4).selfMask().rename('Mask') #Bloom Threshold 3.4 mg/m3
    return img.addBands(mask)

chl_bloom = chl_monthly.map(bloom_mask)

def label_objects(img):
    labels = img.select('Mask').connectedComponents(
        ee.Kernel.square(1), 500
    )
    return img.addBands(labels)

chl_labeled = chl_bloom.map(label_objects)

def bloom_mean(img):
    mean = img.reduceConnectedComponents(
        reducer=ee.Reducer.mean(),
        labelBand='labels'
    )
    return img.addBands(mean.rename('mean'))

chl_mean = chl_labeled.map(bloom_mean)

In [8]:
# MODIS SST variable
sst = ee.ImageCollection('NASA/OCEANDATA/MODIS-Aqua/L3SMI') \
    .select('sst') \
    .filterDate('2003-01-01', '2020-12-31') \
    .map(lambda i: i.clip(roi))

sst_monthly = monthly_composite(sst)
sst_ts = sst_monthly.map(add_time_vars)

chl_ts = chl_mean.map(add_time_vars)

In [9]:
# De-trending
chl_detrended = detrend(chl_ts, 'mean')
sst_detrended = detrend(sst_ts, 'sst')

In [10]:
# Lagged Correlation
def lagged_join(left, right, lag_days):
    filt = ee.Filter.And(
        ee.Filter.maxDifference(
            difference=lag_days * 24 * 60 * 60 * 1000,
            leftField=timeField,
            rightField=timeField
        ),
        ee.Filter.greaterThan(leftField=timeField, rightField=timeField)
    )

    return ee.Join.saveAll(
        matchesKey='images',
        ordering=timeField,
        ascending=False
    ).apply(left, right, filt)


lagged = lagged_join(chl_detrended, sst_detrended, 31)

def merge(img):
    imgs = ee.ImageCollection.fromImages(img.get('images'))
    return imgs.iterate(lambda c, p: ee.Image(p).addBands(c), img)

merged = ee.ImageCollection(lagged.map(merge))

cov = merged.select(['mean', 'sst']) \
    .map(lambda i: i.toArray()) \
    .reduce(ee.Reducer.covariance())

def correlation(vc):
    covar = vc.arrayGet([0, 1])
    sd0 = vc.arrayGet([0, 0]).sqrt()
    sd1 = vc.arrayGet([1, 1]).sqrt()
    return covar.divide(sd0).divide(sd1).rename('correlation')

corr_sst = correlation(cov)

In [12]:
# Export
geemap.ee_export_image_to_drive(
    corr_sst,
    description='crosscor_SST',
    folder='REGRESSION',
    region=roi,
    scale=4000
)